## Load Silver table 

In [0]:
import pyspark.sql.functions as F

silver_users_df        = spark.table("jrvs_databricks_fundamentals.silver.users_data")
silver_cards_df        = spark.table("jrvs_databricks_fundamentals.silver.cards_data")
silver_transactions_df = spark.table("jrvs_databricks_fundamentals.silver.transactions_data")
silver_mcc_df          = spark.table("jrvs_databricks_fundamentals.silver.mcc_codes")
silver_fraud_df        = spark.table("jrvs_databricks_fundamentals.silver.fraud_labels")

Gold Table

In [0]:
gold_transactions_df = (
    silver_transactions_df
    # dimensions — LEFT so no transaction is ever dropped
    .join(silver_users_df, "user_id", "left")
    .join(silver_cards_df, "card_id", "left")
    .join(silver_mcc_df,   "mcc_code", "left")
    .join(silver_fraud_df, "transaction_id", "left")
    # Deal with nulls
    .withColumn(
        "mcc_description",
        F.coalesce(F.col("mcc_description"), F.lit("Unclassified"))
    )
    .withColumn(
        "errors",
        F.when(F.col("errors").isNull(), "No error").otherwise(F.col("errors"))
    )

    .withColumn(
        "fraud_status",
        F.when(F.col("is_fraud") == True,  "Fraud")
         .when(F.col("is_fraud") == False, "Legitimate")
         .otherwise("Unlabeled")
    )
)

## Write to Gold Table

In [ ]:
gold_transactions_df.write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("jrvs_databricks_fundamentals.gold.transactions_data")